# Streamlit app building

## What it will show and how

* Filters can be on: rating, publication year, number of pages. 
* Selection-tick box can be done by: List name, Book source
* Actual book groups recommendations can be done on two eligibility flags. Two different questions, not to be collapsed.

| Flag | Question it answers | App behaviour when `False` |
|---|---|---|
| `clusterable` | Was this book eligible for genre clustering? | Exclude from "more like this" recommendations |
| `mood_scored` | Did this book have a usable synopsis to score? | Exclude from the mood filter |

A book can be `clusterable=True, mood_scored=False`, or the reverse. Branch on each flag
**explicitly** — never infer eligibility from a null cluster or a null mood.

In [9]:
import pandas as pd

df = pd.read_csv("../data/clean/catalog_mood.csv")
print(df.info())
display(df.head(1))


<class 'pandas.DataFrame'>
RangeIndex: 1077 entries, 0 to 1076
Data columns (total 29 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   book_name          1077 non-null   str    
 1   author             1077 non-null   str    
 2   book_url           1077 non-null   str    
 3   img_url            1077 non-null   str    
 4   rating             1077 non-null   float64
 5   pages              1077 non-null   float64
 6   pub_date           1077 non-null   int64  
 7   genres             1077 non-null   str    
 8   genres_clean       1077 non-null   str    
 9   synopsis           1077 non-null   str    
 10  data_source        1077 non-null   str    
 11  source_list        1077 non-null   str    
 12  cluster_f1b        612 non-null    float64
 13  cluster_name       612 non-null    str    
 14  clusterable        1077 non-null   bool   
 15  mood_scored        1077 non-null   bool   
 16  mood               1055 non-null   

,book_name,author,book_url,img_url,rating,pages,pub_date,genres,genres_clean,synopsis,...,emotion_secondary,emotion_conf,mood_low_conf,emo_anger,emo_disgust,emo_fear,emo_joy,emo_neutral,emo_sadness,emo_surprise
0,Persuasion,Jane Austen,https://play.google.com/store/books/details?id...,http://books.google.com/books/content?id=UrrTE...,4.12,260.0,2026,Fiction,['fiction'],Discover the timeless allure of love and secon...,...,joy,0.66172,False,0.004315,0.01351,0.003206,0.285717,0.66172,0.02295,0.008581


In [10]:
df.columns

Index(['book_name', 'author', 'book_url', 'img_url', 'rating', 'pages',
       'pub_date', 'genres', 'genres_clean', 'synopsis', 'data_source',
       'source_list', 'cluster_f1b', 'cluster_name', 'clusterable',
       'mood_scored', 'mood', 'mood_secondary', 'emotion', 'emotion_secondary',
       'emotion_conf', 'mood_low_conf', 'emo_anger', 'emo_disgust', 'emo_fear',
       'emo_joy', 'emo_neutral', 'emo_sadness', 'emo_surprise'],
      dtype='str')

In [13]:
df.source_list.unique()

<ArrowStringArray>
[                              'Best Books Ever',
                       '20th Century Literature',
                                   'Young Adult',
                         'Books You Should Read',
               'The Best Epic Fantasy (fiction)',
                             'Best Epic Fantasy',
         'Books That Should Be Made Into Movies',
                          'Best Book Boyfriends',
                        'Best Young Adult Books',
   'Best Dystopian and Post-Apocalyptic Fiction',
                   'Books That Should Be Movies',
                       'Best Historical Fiction',
 'Books That Everyone Should Read At Least Once',
               'Best Books of the Decade: 2000s',
                               'featured Google',
                'Best Books of the 20th Century']
Length: 16, dtype: str

In [11]:
app_cols= ['book_name', 'author', 'book_url', 'img_url', 'rating', 'pages',
       'pub_date','genres_clean', 'synopsis', 'data_source',
       'source_list', 'cluster_f1b', 'cluster_name', 'clusterable',
       'mood_scored', 'mood', 'mood_secondary', 'emotion', 'emotion_secondary',
       'emotion_conf', 'mood_low_conf']

app_df=df.copy()
app_df= app_df[app_cols]
app_df.head(1)

,book_name,author,book_url,img_url,rating,pages,pub_date,genres_clean,synopsis,data_source,...,cluster_f1b,cluster_name,clusterable,mood_scored,mood,mood_secondary,emotion,emotion_secondary,emotion_conf,mood_low_conf
0,Persuasion,Jane Austen,https://play.google.com/store/books/details?id...,http://books.google.com/books/content?id=UrrTE...,4.12,260.0,2026,['fiction'],Discover the timeless allure of love and secon...,google_books,...,NaN,NaN,False,True,Contemplative,Uplifting,neutral,joy,0.66172,False


In [ ]:
import streamlit as st
import time

df = pd.read_csv("../data/clean/catalog_mood.csv")
app_cols= ['book_name', 'author', 'book_url', 'img_url', 'rating', 'pages',
       'pub_date','genres_clean', 'synopsis', 'data_source',
       'source_list', 'cluster_f1b', 'cluster_name', 'clusterable',
       'mood_scored', 'mood', 'mood_secondary', 'emotion', 'emotion_secondary',
       'emotion_conf', 'mood_low_conf']

app_df=df.copy()
app_df= app_df[app_cols]
app_df.head(1)

# --- 2. LAYOUT ---
st.write("Aloha my dear reader! Ready to blow your eyes with some Book recommendations?")
st.header("WELCOME TO Words4U App")
st.write("Your right Book discovery Engine")

# Interactive widget to greet reader
name = st.text_input("How do you want to be called today? or the boring question, what's your name?")
if name:
    st.write(f"Hello hello, {name}! 👋")
else:
    st.write("Hello hello, anonymous friend! 👋")

# --- 3. SIDEBAR / FILTERING ---
# Safe extraction of unique moods
moods = sorted(app_df.loc[app_df["mood_scored"] == True, "mood"].dropna().unique())
selected = st.selectbox("What mood are you in?", ["Any"] + list(moods))

# Filter the DataFrame based on selection
filtered_df = app_df.copy()
if selected != "Any":
    filtered_df = filtered_df[filtered_df["mood"] == selected]

# --- 4. COLUMNS ---
st.subheader("Current Recommendations")
col1, col2, col3, col4 = st.columns(4)
col1.write("**Sr no.**")
col2.write("**Book Title**")
col3.write("**Author Name**")
col4.write("**Year Published**")

# Loop through the filtered dataframe to populate the table dynamically
for index, row in filtered_df.reset_index(drop=True).iterrows():
    col1.write(f"{index + 1}")
    col2.write(row["book_title"])
    col3.write(row["author"])
    col4.write(f"{row['pub_date']}")

# --- 5. TABS ---
st.write("---")
tab1, tab2 = st.tabs(["Book synopsis", "Another book alike"])

# Ensure we have data left after filtering to avoid index errors
if not filtered_df.empty:
    # Always safe to get the first book
    first_book = filtered_df.iloc[0]
    
    with tab1:
        st.subheader(f"Read more about {first_book['book_title']}")
        # Safe lookup: Check if the 'synopsis' column exists in the row, otherwise use a fallback string
        synopsis = first_book.get('synopsis', 'No synopsis available.')
        st.write(synopsis)
        st.image(first_book['img_url'], width=150)
        
    with tab2:
        # Check if we actually have at least 2 books in the filtered results
        if len(filtered_df) >= 2:
            second_book = filtered_df.iloc[1]
            st.subheader(f"Read books alike {second_book['book_title']}")
            
            # Safe lookup using the second_book Series object
            second_synopsis = second_book.get('synopsis', 'No synopsis available.')
            st.write(second_synopsis)
            
            # Note: Did you mean to use second_book['img_url'] here instead of first_book?
            st.image(second_book['img_url'], width=150)
        else:
            st.subheader("Looking for similar books?")
            st.info("Only one book matches this mood! No similar recommendations are available right now.")
else:
    st.warning("No books match your selected mood!")